# When the document is wrong: do LLMs notice? (Colab runner)

1. **Runtime → Change runtime type → A100 GPU** (an L4 also works, but skips the 14B model).
2. Run the cells from top to bottom.

New runs are written to `my_results/`, so they never mix with the published results in `results/`. Results are saved
after every chunk: if Colab disconnects, reconnect, run cells 1–4 again and re-run the cell that stopped; finished
questions are skipped automatically. Keep this browser tab open while it runs, and download `my_results/` before you
close the session (files in `/content` are deleted when the runtime ends).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/SoroushJaberi/rag-knowledge-conflict.git /content/rag-knowledge-conflict
%cd /content/rag-knowledge-conflict
!ls

In [ ]:
# Installs vLLM (fast inference) and the data tools. Takes a few minutes. If Colab asks you to restart the session,
# restart, then run cell 2 again and continue from cell 4 (no need to re-install).
!pip install -q vllm datasets nvidia-ml-py

In [ ]:
# Rebuild the 1,000-question evaluation set from NQ-Swap and check it matches the published one
!python prepare_data.py
!sha256sum data/items.jsonl
print("expected: f0f368fb7c83ec2f5a2ded51c2d268315db262ced27242436331dd38987a6e27")

### Check the code, then a 5-minute smoke test with the smallest model

In [ ]:
!python test_scoring.py
!python run_eval.py --model Qwen/Qwen3-0.6B --think 0 --limit 10 --out smoke
!python run_eval.py --model Qwen/Qwen3-0.6B --think 1 --limit 5 --out smoke
!head -n 3 smoke/Qwen3-0.6B_nothink/predictions.jsonl

import torch
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
RUNS = [("Qwen/Qwen3-1.7B", 0), ("Qwen/Qwen3-1.7B", 1),
        ("Qwen/Qwen3-4B", 0),   ("Qwen/Qwen3-4B", 1),
        ("Qwen/Qwen3-8B", 0),   ("Qwen/Qwen3-8B", 1),
        ("Qwen/Qwen3-14B", 0),  ("Qwen/Qwen3-14B", 1)]
for model, think in RUNS:
    if "14B" in model and gpu_gb < 35:
        print("skipping", model, "(needs an A100 40GB)")
        continue
    !python run_eval.py --model {model} --think {think} --out my_results

In [ ]:
import torch
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
RUNS = [("Qwen/Qwen3-1.7B", 0), ("Qwen/Qwen3-1.7B", 1),
        ("Qwen/Qwen3-4B", 0),   ("Qwen/Qwen3-4B", 1),
        ("Qwen/Qwen3-8B", 0),   ("Qwen/Qwen3-8B", 1),
        ("Qwen/Qwen3-14B", 0),  ("Qwen/Qwen3-14B", 1)]
for model, think in RUNS:
    if "14B" in model and gpu_gb < 35:
        print("skipping", model, "(needs an A100 40GB)")
        continue
    !python run_eval.py --model {model} --think {think}

import torch
if torch.cuda.get_device_properties(0).total_memory / 1e9 > 75:
    for think in (0, 1):
        !python run_eval.py --model Qwen/Qwen3-32B --think {think} --out my_results
else:
    print("32B skipped: this GPU has less than 80 GB")

In [ ]:
import torch
if torch.cuda.get_device_properties(0).total_memory / 1e9 > 75:
    for think in (0, 1):
        !python run_eval.py --model Qwen/Qwen3-32B --think {think}
else:
    print("32B skipped: this GPU has less than 80 GB")

!python analyze.py --results my_results
from IPython.display import Image, Markdown, display
display(Markdown(open('my_results/summary.md', encoding='utf-8').read()))
for f in ['overwrite_vs_size.png', 'conflict_auroc_vs_size.png', 'outcomes_by_condition.png']:
    display(Image('my_results/figures/' + f))
# Download: !zip -r -q my_results.zip my_results  then  from google.colab import files; files.download('my_results.zip')

In [ ]:
!python analyze.py
from IPython.display import Image, Markdown, display
display(Markdown(open('results/summary.md', encoding='utf-8').read()))
for f in ['overwrite_vs_size.png', 'conflict_auroc_vs_size.png', 'outcomes_by_condition.png']:
    display(Image('results/figures/' + f))